In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

2.6.0+cu124
True
12.4
NVIDIA RTX A6000


In [ ]:
!pip install langchain langchain-community langchain-core
!pip install langchain-huggingface
!pip install langchain-groq
!pip install faiss-cpu
!pip install sentence-transformers
!pip install python-dotenv
!pip install langchain_ollama
!pip install langchain_unstructured
!pip install langchain_community
!pip install langchain_text_splitters
!pip install langchain_experimental

^C


In [ ]:
from langchain_ollama.llms import OllamaLLM
from langchain_unstructured import UnstructuredLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.documents import Document # Added import for Document
from langchain_groq import ChatGroq
import os
# import streamlit as st
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

In [4]:
load_dotenv()
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [ ]:
import os
groq_api_key = os.getenv("GROQ_API_KEY", "your-key-here")

In [6]:
# Create an Ollama Client using langChain's Ollama Component
llm = ChatGroq(
  model="llama-3.1-8b-instant",
  temperature=0.5,
  max_retries=2
)

In [7]:
print(llm.invoke("Hello!"))

content='Hello. How can I assist you today?' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 37, 'total_tokens': 47, 'completion_time': 0.010360646, 'completion_tokens_details': None, 'prompt_time': 0.00240642, 'prompt_tokens_details': None, 'queue_time': 0.035999271, 'total_time': 0.012767066}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d69d5-5c42-7a43-bb7a-e4d509234a46-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 37, 'output_tokens': 10, 'total_tokens': 47}


In [ ]:
law_file_paths = [
    r"C:\Users\yrghimire\Chapter\spring_2026\comp841\code\llm-agentic-legal-information-retrieval\laws_de.csv"
]

court_file_paths = [
    r"C:\Users\yrghimire\Chapter\spring_2026\comp841\code\llm-agentic-legal-information-retrieval\court_considerations.csv"
]
#court_considerations.csv

In [ ]:
import pandas as pd
from langchain_core.documents import Document # Added for self-sufficiency

# real_documents = []

# for file_path in file_paths:
#     if file_path.endswith(".csv"):
#         try:
#             df = pd.read_csv(
#                 file_path,
#                 on_bad_lines="skip",
#                 engine="python" # Using 'python' engine for better error handling
#             )

#             for _, row in df.iterrows():

#                 metadata = row.to_dict()

#                 # Check if 'text' and 'citation' keys exist in the metadata
#                 if "text" in metadata and "citation" in metadata:
#                     real_documents.append(
#                         Document(
#                             page_content=str(metadata["text"]),   # main legal text
#                             metadata={
#                                 "citation": str(metadata["citation"])
#                             }
#                         )
#                     )
#                 else:
#                     # Fallback if 'text' or 'citation' are not present in a row
#                     # Use all available data as page_content and original metadata
#                     page_content = " ".join([str(v) for v in row.values if pd.notna(v)])
#                     real_documents.append(
#                         Document(
#                             page_content=page_content,
#                             metadata=metadata
#                         )
#                     )

#         except Exception as e:
#             print(f"Error processing {file_path}: {e}")

#     else:
#         print(f"Unsupported file type: {file_path}")

# print(f"Successfully loaded {len(real_documents)} documents.")

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 64
    }
)

from tqdm.auto import tqdm

def build_faiss_streaming(file_paths, save_path= r"C:\Users\yrghimire\Chapter\spring_2026\comp841\code\data"):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100
    )

    vectorstore = None

    # File-level progress
    for file_path in tqdm(file_paths, desc="Processing files"):

        if not file_path.endswith(".csv"):
            continue

        try:
            # Chunk-level progress
            chunk_iter = pd.read_csv(
                file_path,
                chunksize=800,
                on_bad_lines="skip",
                engine="python"
            )

            for chunk_df in tqdm(chunk_iter, desc=f"Reading {file_path}", leave=False):

                docs_batch = []

                for row in chunk_df.itertuples(index=False):
                    metadata = row._asdict()

                    if hasattr(row, "text") and hasattr(row, "citation"):
                        docs_batch.append(
                            Document(
                                page_content = "passage: " + str(row.text),
                                metadata={"citation": str(row.citation)}
                            )
                        )
                    else:
                        page_content = " ".join(
                            [str(v) for v in metadata.values() if pd.notna(v)]
                        )
                        docs_batch.append(
                            Document(
                                page_content=page_content,
                                metadata=metadata
                            )
                        )

                # Splitting progress
                split_docs = text_splitter.split_documents(docs_batch)

                #  Embedding + FAISS progress
                if vectorstore is None:
                    print("Building FAISS index...")
                    vectorstore = FAISS.from_documents(split_docs, embedding_model)
                else:
                    vectorstore.add_documents(split_docs)

        except Exception as e:
            print(f" Error processing {file_path}: {e}")

    if vectorstore:
        print(" Saving FAISS index...")
        vectorstore.save_local(save_path)

    print(" Done!")
    return vectorstore

C:\Users\yrghimire\AppData\Local\Temp\ipykernel_11976\2415155453.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

c:\Users\yrghimire\AppData\Local\miniconda3\envs\legal_rag\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yrghimire\.cache\huggingface\hub\models--intfloat--multilingual-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [11]:
from langchain_core.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents and an optional parameter k used in the RRF formula """

    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

In [12]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [13]:
def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

In [ ]:
law_file_paths = [
    r"C:\Users\yrghimire\Chapter\spring_2026\comp841\code\llm-agentic-legal-information-retrieval\laws_de.csv"
]

court_file_paths = [
    r"C:\Users\yrghimire\Chapter\spring_2026\comp841\code\llm-agentic-legal-information-retrieval\court_considerations.csv"
]

real_vector_law_store = build_faiss_streaming(law_file_paths, save_path= r"C:\Users\yrghimire\Chapter\spring_2026\comp841\code\data\law_vectorstore")

real_vector_court_store = build_faiss_streaming(court_file_paths, save_path= r"C:\Users\yrghimire\Chapter\spring_2026\comp841\code\data\court_vectorstore")
real_vector_court_store = build_faiss_streaming(court_file_paths, save_path= r"C:\Users\yrghimire\Chapter\spring_2026\comp841\code\data\court_vectorstore")


NameError: name 'law_file_paths' is not defined

In [17]:
retriever = real_vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 5})

In [18]:
template = """You are a legal citation retrieval agent. Your task is to analyze a case statement and retrieve citations from the provided legal documents. Documents may appear in multiple languages (e.g., German or Swiss languages), so translation or cross-lingual understanding may be necessary. Documents contain 'citation' as metadata and 'text' as page_content. Look through the text, if relevant than extract it's citation and append to output.

Input: {question}
Output: A semicolon-separated list of the most relevant Swiss legal citations only."""
prompt_perspectives = PromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser

generate_query = (
    prompt_perspectives
    | llm # Reusing the pre-configured llm object
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

In [ ]:
import pandas as pd

# Load validation dataset
queries_df = pd.read_csv(r"C:\Users\yrghimire\Chapter\spring_2026\comp841\code\llm-agentic-legal-information-retrieval\val.csv")

results = []

for _, row in queries_df.iterrows():

    query_id = row["query_id"]
    question = row["query"]
    gold = row["gold_citations"]

    try:
        # Retrieval pipeline
        retrieval_chain = generate_query | retriever.map() | get_unique_union
        docs = retrieval_chain.invoke({"question": question})

        # Extract predicted citations
        citations = []
        for doc in docs:
            if "citation" in doc.metadata:
                citations.append(doc.metadata["citation"])

        citations = list(set(citations))
        predicted = ",".join(citations)

    except Exception as e:
        print(f"Error processing {query_id}: {e}")
        predicted = ""

    results.append({
        "query_id": query_id,
        "query": question,
        "gold_citations": gold,
        "predicted_citations": predicted
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Save predictions
results_df.to_csv(r"C:\Users\yrghimire\Chapter\spring_2026\comp841\code\data\val_predictions.csv", index=False)

print("Predictions saved.")
print(results)


OSError: Cannot save file into a non-existent directory: '\kaggle\working'

In [ ]:
def compute_metrics(row):

    gold = set(str(row["gold_citations"]).split(","))
    pred = set(str(row["predicted_citations"]).split(","))

    if "" in gold:
        gold.remove("")
    if "" in pred:
        pred.remove("")

    tp = len(gold & pred)

    precision = tp / len(pred) if len(pred) > 0 else 0
    recall = tp / len(gold) if len(gold) > 0 else 0

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return pd.Series([precision, recall, f1])


results_df[["precision","recall","f1"]] = results_df.apply(compute_metrics, axis=1)

print("Average Precision:", results_df["precision"].mean())
print("Average Recall:", results_df["recall"].mean())
print("Average F1:", results_df["f1"].mean())

In [ ]:
queries_df = pd.read_csv("/kaggle/input/competitions/llm-agentic-legal-information-retrieval/test.csv")

results = []

for _, row in queries_df.iterrows():

    query_id = row["query_id"]
    question = row["query"]
    # gold = row["gold_citations"]

    try:
        # Retrieval pipeline
        retrieval_chain = generate_query | retriever.map() | get_unique_union
        docs = retrieval_chain.invoke({"question": question})

        # Extract predicted citations
        citations = []
        for doc in docs:
            if "citation" in doc.metadata:
                citations.append(doc.metadata["citation"])

        citations = list(set(citations))
        predicted = ",".join(citations)

    except Exception as e:
        print(f"Error processing {query_id}: {e}")
        predicted = ""

    results.append({
        "query_id": query_id,
        # "query": question,
        # "gold_citations": gold,
        "predicted_citations": predicted
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Save predictions
results_df.to_csv("/kaggle/working/test_submission.csv", index=False)

print("Predictions saved.")
print(results)
